In [1]:
# ----------------------------------------------------------------------
# 라이브러리 및 데이터 준비
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# 정상 데이터 포인트 (대부분 10 주변에 집중)
rng = np.random.RandomState(42)
X = 0.3 * rng.randn(100, 2)
X_train = np.r_[X + 2, X - 2] # 2개의 군집 생성

# 명확한 이상치 추가 (데이터셋에서 멀리 떨어진 지점)
X_outliers = rng.uniform(low=-4, high=4, size=(20, 2))
X_data = np.r_[X_train, X_outliers] # 학습 데이터 + 이상치

df = pd.DataFrame(X_data, columns=['Feature1', 'Feature2'])
print("--- 원본 데이터셋 (정상 + 이상치) ---")
print(df.head())

--- 원본 데이터셋 (정상 + 이상치) ---
   Feature1  Feature2
0  2.149014  1.958521
1  2.194307  2.456909
2  1.929754  1.929759
3  2.473764  2.230230
4  1.859158  2.162768


In [2]:
# ----------------------------------------------------------------------
# 2단계: Isolation Forest 모델 학습
# ----------------------------------------------------------------------
# contamination=0.15: 전체 데이터의 약 15%가 이상치라고 가정하고 모델을 학습 (총 240개 중 20개의 이상치, 약 8.3%지만 예시로 15% 사용)
model = IsolationForest(contamination=0.15, random_state=42)

# 모델 학습 (X_data의 패턴을 학습)
model.fit(X_data)

print("\n--- 2단계: 모델 학습 완료 ---")


--- 2단계: 모델 학습 완료 ---


In [3]:
# ----------------------------------------------------------------------
# 3단계: 이상치 예측 및 점수 확인
# ----------------------------------------------------------------------

# 1) 예측: 각 데이터 포인트에 대해 이상치 여부 예측
#    결과: 1 (정상), -1 (이상치)
predictions = model.predict(X_data)

# 2) 이상치 점수 (Anomaly Score) 계산
#    낮은 점수일수록 이상치일 확률이 높음
scores = model.decision_function(X_data)

# 결과를 DataFrame에 추가
df['anomaly_prediction'] = predictions
df['anomaly_score'] = scores

print("\n--- 3단계: 예측 및 점수 결과 (상위 5개) ---")
print(df.head()) # 대부분 1(정상)과 높은 점수(덜 이상함)


--- 3단계: 예측 및 점수 결과 (상위 5개) ---
   Feature1  Feature2  anomaly_prediction  anomaly_score
0  2.149014  1.958521                   1       0.088736
1  2.194307  2.456909                   1       0.037808
2  1.929754  1.929759                   1       0.096795
3  2.473764  2.230230                   1       0.009035
4  1.859158  2.162768                   1       0.078680


In [4]:
# ----------------------------------------------------------------------
# 4단계: 이상치 필터링 및 확인
# ----------------------------------------------------------------------
# 예측값이 -1인 데이터만 필터링하여 이상치 확인
outliers = df[df['anomaly_prediction'] == -1].sort_values(by='anomaly_score')

print("\n--- 4단계: 탐지된 이상치 확인 (점수가 낮은 순 = 가장 이상함) ---")
print(outliers.head())

# 해석: anomaly_score가 0에 가까울수록(또는 음수일수록) 이상치일 가능성이 높고,
#       predictions가 -1이면 모델이 이상치로 분류했음을 의미합니다.


--- 4단계: 탐지된 이상치 확인 (점수가 낮은 순 = 가장 이상함) ---
     Feature1  Feature2  anomaly_prediction  anomaly_score
213 -3.755998 -3.701214                  -1      -0.300176
204  2.936579  3.305924                  -1      -0.252973
218 -3.586546  0.250837                  -1      -0.244863
210  0.626241 -3.712462                  -1      -0.233414
208  3.120043 -1.296039                  -1      -0.232923
